In [1]:
from astropy.cosmology import Planck18
#from zdm import cosmology as cos
from zdm.zdm import misc_functions
from zdm.zdm import parameters
#from zdm import survey
#from zdm import pcosmic
#from zdm import iteration as it
from zdm.zdm import loading
#from zdm import io
import matplotlib.pyplot as plt

import numpy as np
#from matplotlib import pyplot as plt
from pkg_resources import resource_filename
#import time

from frb.dm import igm
from frb.scripts.pzdm_mag import main as pzdm_mag_main
import argparse
import pandas as pd
from scipy.stats import spearmanr

/Users/lmasriba/FRBs/frb/halos/hmf.py:51: UserWarning: hmf_emulator not imported.  Hope you are not intending to use the hmf.py module..
  warnings.warn("hmf_emulator not imported.  Hope you are not intending to use the hmf.py module..")


In [ ]:
#params 
nmcs = 1000 
nu = 1. # GHz
t_samp = 1.182  # ms
bandwidth = 1.0 # MHz
snrthreshold = 10. # SNR threshold
mean_DM_MW = 80. # pc cm^-3
disp_DM_MW = 50. # pc cm^-3



survey_path ='/Users/lmasriba/FRBs/zdm/zdm/data/Surveys/CHIME/' # this needs to be changed to the path where the survey files are located on your machine

In [ ]:
def htr(file):
        """ upload data from Scott+2025 """
        df = pd.read_csv(file, delim_whitespace=True)
        #print (df.columns)
        return df


def set_state():
    """
    Initializes the state without the contribution from the host DM - uses best fit parameters from Hoffman+2025.
    """

    param_dict={'lmean': 0.01, 'lsigma': 0.42, , 'Wlogmean': -1,'WNbins': 1,'Wlogsigma': 0.1, 'Slogmean': -2,'Slogsigma': 0.1, # these essentially turn off DM host and set all FRB widths to ~0 (or close enough)
        # next the parameters that control the FRB population and cosmology
        'sfr_n': 0.21, 'alpha': 0.11,'lEmax': 41.37, 'lEmin': 39.47, 'gamma': -1.04, 
        'H0': 70.23, 'halo_method': 0, 'sigmaDMG': 0.0, 'sigmaHalo': 0.0,
        'lC': -7.61, 'min_lat': 0.0}
    
    state = parameters.State()
    state.set_astropy_cosmo(Planck18)
    state.update_params(param_dict)

    # Initialize with state parameters
    #cos.set_cosmology(state.params)
    #cos.init_dist_measures()
    return state

def creategrid(state):

    # defines CHIME grids to load
    NDECBINS=6
    cnames=[]
    for i in np.arange(NDECBINS):
        cname="CHIME_decbin_"+str(i)+"_of_6"
        cnames.append(cname)
    survey_dir = survey_path

    for it,nz in enumerate(powind):
        state = set_state(nz)

        css,cgs = loading.surveys_and_grids(survey_names=cnames, init_state=state, rand_DMG=False,sdir = survey_dir, repeaters=True)

        # compiles sums over all six declination bins
        crates = cgs[0].rates * 10**cgs[0].state.FRBdemo.lC * css[0].TOBS
        creps = cgs[0].exact_reps * cgs[0].state.rep.RC
        csingles = cgs[0].exact_singles * cgs[0].state.rep.RC
        
        for i,g in enumerate(cgs):
            s = css[i]
            if i ==0:
                continue
            else:
                crates += g.rates * 10**g.state.FRBdemo.lC * s.TOBS
                creps += g.exact_reps * g.state.rep.RC
                csingles += g.exact_singles * g.state.rep.RC

    # This needs to be checked -- to make sure it rewrites cgs[0].rates (also for reps and singles) to the sum over all six declination bins, which is what we want to plot
    cgs[0].rates = crates 
    cgs[0].exact_reps = creps
    cgs[0].exact_singles = csingles

    return cgs[0]

    # in case you wish to switch to another output directory
    #opdir = "../ScatSim/"
    #if not os.path.exists(opdir):
    #    os.mkdir(opdir)
    
    # Initialise surveys and grids
    #sdir = os.path.join(resource_filename('zdm', 'data'), 'Surveys')
    sdir = '/Users/lmasriba/FRBs/zdm/zdm/data/Surveys'
    
    names = ["CRAFT_ICS_1300"]
    
    # essentially turns off DM host and sets all FRB widths to ~0 (or close enough)
    param_dict = {'lmean': 0.01, 'lsigma': 0.4, 'Wlogmean': -1,'WNbins': 1,
        'Wlogsigma': 0.1, 'Slogmean': -2,'Slogsigma': 0.1}
    state = parameters.State()
    state.set_astropy_cosmo(Planck18)
    state.update_params(param_dict)
    
    
    surveys, grids = loading.surveys_and_grids(init_state=state, survey_names = names,
        repeaters=False, sdir=sdir)
    
    g = grids[0]

In [ ]:
main()

    state = set_state()
    creategrid(state)

